In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, concat_ws, date_format, to_date, current_timestamp, lit, expr, month
from pyspark.sql.types import LongType

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df1 = spark.read.parquet("/user/sclsepb4532/sclsepb4532_staging.db/order_n_customer", header=True, inferSchema=True)
df2 = spark.read.parquet("/user/sclsepb4532/sclsepb4532_staging.db/payments_n_order", header=True, inferSchema=True)
df3 = spark.read.parquet("/user/sclsepb4532/sclsepb4532_staging.db/order_items_n_products", header=True, inferSchema=True)

26/11/08 14:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 14:04:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/08 14:04:44 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/08 14:06:40 WARN DataStreamer: Slow waitForAckedSeqno took 49997ms (threshold=30000ms). File being written: /user/sclsepb4532/.sparkStaging/application_1793087731018_0433/__spark_libs__176208372458342931.zip, block: BP-1364978691-172.31.64.10-1677754292516:blk_1101625937_27893833, Write pipeline datanodes: [DatanodeInfoWithStorage[172.31.64.13:9866,DS-e595a26d-2fb4-4683-b8bf-ad5a6b69a54b,DISK], DatanodeInfoWithStorage[172.31.64.11:9866,DS-8af7bd92-c7a4-44fb-b361-00496b8456c3,DI

In [2]:
df = df1.join(df2, df1['order_id']==df2['order_id'],'inner')
df = df.drop(df2['order_id'])
df = df.join(df3, df['order_id']==df3['order_id'],'inner')
df = df.drop(df3['order_id'])
df = df.drop(df1['customer_id'])
df = df.drop(df1['order_channel'])
df = df.drop(df1['store_id'])
df = df.drop(df1['order_date'])
df = df.drop(df1['order_status'])

In [3]:
transformed_sales_df = df.select(
    expr("abs(hash(order_id, product_id))").cast(LongType()).alias("sales_key"),    
    col("order_id"),
    sha2(col("customer_id"), 256).alias("customer_key"),
    sha2(col("product_id"), 256).alias("product_key"),
    expr("CASE WHEN order_channel = 'Online' OR store_id IS NULL THEN NULL ELSE sha2(store_id, 256) END").alias("store_key"),
    date_format(to_date(col("order_date"), "yyyy-MM-dd"), "yyyyMMdd").cast("int").alias("date_key"),
    month(to_date(col("order_date"), "yyyy-MM-dd")).alias("order_month"),
    col("order_channel"),
    col("order_status"),
    col("payment_status"),
    col("quantity").cast("int"),
    (col("quantity") * col("item_price")).cast("double").alias("gross_amount"),
    col("discount_amount").cast("double"),
    ((col("quantity") * col("item_price")) - col("discount_amount")).cast("double").alias("net_amount"),
    col("payment_method"),
    current_timestamp().alias("load_timestamp")
)

final_sales_df = transformed_sales_df.dropDuplicates(["order_id", "product_key"])

In [4]:
#target_table_name = "sclsepb4532_analytics.fact_sales_analytics"
final_sales_df.write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .partitionBy("order_month") \
        .saveAsTable("sclsepb4532_analytics.fact_sales_analytics")

26/11/08 14:07:10 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/08 14:32:48 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
